#1. Imports And Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Ensure the current directory is in the system path so Python can find your .py files
sys.path.append(os.path.abspath('.'))

# 2. Import your custom-built modules
# NB - These must be uploaded to your Colab session for this to run!
try:
    from data_ingestion import create_db_engine, read_from_sql
    from field_data_processor import FieldDataProcessor
    # from validate_data import some_validation_function # Add if you have a specific test function
    print("✅ Custom modules imported successfully.")
except ImportError as e:
    print(f"❌ Error: Could not find your module files. {e}")
    print("Check if data_ingestion.py and field_data_processor.py are uploaded to Colab.")

# 3. Visual settings
%matplotlib inline
sns.set_theme(style="whitegrid")

#2. Load And Clean

In [ ]:
# This is where we tell our code exactly what to do
config_params = {
    "db_path": 'sqlite:///Maji_Ndogo_farm_survey_small.db',
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features USING (Field_ID)
        LEFT JOIN soil_and_crop_features USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,
    "columns_to_rename": {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'},
    "values_to_rename": {'cassaval': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'},
    "regex_patterns" : {
            'Rainfall': r'(\d+(\.\d+)?)\s?mm',
            'Temperature': r'(\d+(\.\d+)?)\s?C',
            'Pollution_level': r'=\s*(-?\d+(\.\d+)?)|Pollution at \s*(-?\d+(\.\d+)?)'
            }
}

# 2. Ingest the data from the SQLite database
print("Connecting to database and fetching records...")
engine = create_db_engine(config_params["db_path"])
raw_df = read_from_sql(engine, config_params["sql_query"])

# 3. Clean the data using your FieldDataProcessor module
print("Starting the data cleaning process...")
field_processor = FieldDataProcessor(config_params)
field_processor.process()

# 4. Save the cleaned result to a final dataframe for our Regression model
df = field_processor.df

# 5. Quick check to ensure the data is ready
print(f"✅ Success! Integrated dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

#3. The Linear Regression Model

In [ ]:
# 1. Define Features (X) and Target (y)
# We use the environmental factors cleaned by our modules to predict yield
features = ['Ave_temps', 'Pollution_level', 'Rainfall']
target = 'Standard_yield'

X = df[features]
y = df[target]

# 2. Split the data into Training (80%) and Testing (20%) sets
# This ensures we test the model on data it hasn't seen before
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and Train the Linear Regression Model
print("Training the Linear Regression model...")
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Make Predictions on the Test Set
predictions = model.predict(X_test)

# 5. Calculate Performance Metrics
r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("\n" + "="*30)
print("   MODEL PERFORMANCE RESULTS")
print("="*30)
print(f"R-squared (R²): {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print("="*30)

# 6. Display the Model Coefficients (Weight of each feature)
coeff_df = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
print("\nFeature Impact:")
print(coeff_df)